# Order Fulfillment Data Platform — Phase 2
### Transformation, Validation & Integration Notebook
**Name:** Llerin, Anton Uriel Medilo
**Track:** Data Engineer — Midterm Activity 2
**Date:** September 15, 2026

In [32]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os
import pandas as pd

matches = subprocess.run(
    ["find", "/content/drive", "-iname", "data_engineering_project", "-type", "d"],
    capture_output=True, text=True
).stdout.splitlines()

if not matches:
    raise FileNotFoundError("Could not find 'data_engineering_project' in this Drive.")

PROJECT_PATH = matches[0]
STAGING_DIR = f"{PROJECT_PATH}/staging"
OUTPUT_DIR = f"{PROJECT_PATH}/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Project: {PROJECT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/data_engineering_project


## Task 1 — Load and Recheck the Staging Layer
Load the five `stg_*.csv` files produced in Activity 1. These are read-only inputs for Phase 2 — no raw or staging file is modified in place.

In [33]:
import pandas as pd
import numpy as np


class StagingLoader:
    """Loads staging CSVs and re-checks primary-key integrity before integration."""

    def __init__(self, staging_dir, pk_map):
        self.staging_dir = staging_dir
        self.pk_map = pk_map          # {source_name: primary_key_column}
        self.data = {}                # {source_name: DataFrame}

    def load_all(self):
        for name in self.pk_map:
            self.data[name] = pd.read_csv(f"{self.staging_dir}/stg_{name}.csv")
        return self

    def summary(self):
        rows = []
        for name, df in self.data.items():
            pk = self.pk_map[name]
            rows.append({
                "file_name": f"stg_{name}.csv",
                "row_count": df.shape[0],
                "column_count": df.shape[1],
                "primary_key": pk,
                "pk_nulls": df[pk].isna().sum(),
                "pk_duplicates": df[pk].duplicated().sum(),
            })
        summary_df = pd.DataFrame(rows)
        summary_df["status"] = np.where(
            (summary_df["pk_nulls"] == 0) & (summary_df["pk_duplicates"] == 0),
            "OK", "REVIEW NEEDED"
        )
        return summary_df


pk_map = {
    "customers": "customer_id", "products": "product_id", "orders": "order_id",
    "payments": "payment_id", "deliveries": "delivery_id",
}

loader = StagingLoader(STAGING_DIR, pk_map).load_all()
staging = loader.data          # dict of DataFrames, used by everything downstream
loader.summary()

,file_name,row_count,column_count,primary_key,pk_nulls,pk_duplicates,status
0,stg_customers.csv,15,6,customer_id,0,0,OK
1,stg_products.csv,15,6,product_id,0,0,OK
2,stg_orders.csv,15,6,order_id,0,0,OK
3,stg_payments.csv,15,6,payment_id,0,0,OK
4,stg_deliveries.csv,15,7,delivery_id,0,0,OK


## Task 2 — Apply Required Transformations
Convert date and numeric fields, then build the derived fields (`successful_amount_paid`, `balance_due`, `is_fully_paid`, `has_delivery_record`, `fulfillment_status`). Each business rule is stated explicitly in code comments so it's auditable.

In [34]:
class FulfillmentTransformer:
    """Standardizes types and builds the order-level derived fields."""

    SUCCESSFUL_PAYMENT_STATUSES = ["Completed"]
    VALID_DELIVERY_STATUSES = ["Delivered", "In Transit", "Shipped"]

    DATE_COLS = {
        "customers": ["registration_date"], "orders": ["order_date"],
        "payments": ["payment_date"], "deliveries": ["shipment_date", "delivery_date"],
    }
    NUMERIC_COLS = {
        "products": ["unit_price", "stock_quantity"],
        "orders": ["total_order_value"], "payments": ["amount_paid"],
    }

    def __init__(self, staging):
        self.staging = staging
        self.orders = None

    def standardize_types(self):
        for source, cols in self.DATE_COLS.items():
            for col in cols:
                self.staging[source][col] = pd.to_datetime(self.staging[source][col], errors="coerce")
        for source, cols in self.NUMERIC_COLS.items():
            for col in cols:
                self.staging[source][col] = pd.to_numeric(self.staging[source][col], errors="coerce")
        return self

    def _successful_amount_paid(self):
        payments = self.staging["payments"]
        return (payments[payments["payment_status"].isin(self.SUCCESSFUL_PAYMENT_STATUSES)]
                .groupby("order_id")["amount_paid"].sum()
                .rename("successful_amount_paid"))

    def _has_delivery_record(self):
        deliveries = self.staging["deliveries"]
        return (deliveries.assign(v=deliveries["delivery_status"].isin(self.VALID_DELIVERY_STATUSES))
                .groupby("order_id")["v"].any()
                .rename("has_delivery_record"))

    def build_derived_fields(self):
        orders = self.staging["orders"].copy()
        orders = orders.join(self._successful_amount_paid(), on="order_id")
        orders = orders.join(self._has_delivery_record(), on="order_id")

        orders["successful_amount_paid"] = orders["successful_amount_paid"].fillna(0.0)
        orders["has_delivery_record"] = orders["has_delivery_record"].fillna(False)

        # balance_due not clipped to zero — negative = overpayment, flagged for review
        orders["balance_due"] = orders["total_order_value"] - orders["successful_amount_paid"]
        orders["is_fully_paid"] = orders["balance_due"] <= 0

        orders["fulfillment_status"] = np.select(
            [orders["total_order_value"].isna(), ~orders["is_fully_paid"], ~orders["has_delivery_record"]],
            ["Data Issue", "Payment Issue", "Delivery Pending"],
            default="Complete"
        )
        self.orders = orders
        return self


transformer = FulfillmentTransformer(staging).standardize_types().build_derived_fields()
orders = transformer.orders
orders["fulfillment_status"].value_counts()

,count
fulfillment_status,
Payment Issue,15


## Task 3

In [35]:
class IntegrityChecker:
    """Runs FK checks and the product-linkage check, and builds the data-quality report."""

    def __init__(self, staging):
        self.staging = staging
        self.orphans = {}
        self.report_rows = []

    def check_fk(self, rule_name, child_name, fk_col, parent_name, pk_col):
        child, parent = self.staging[child_name], self.staging[parent_name]
        orphans = child[~child[fk_col].isin(parent[pk_col])]
        self.orphans[rule_name] = orphans
        self.report_rows.append({
            "rule": rule_name, "source": child_name,
            "affected_rows": len(orphans),
            "status_action": "Reported for review; not silently removed",
        })
        return orphans

    def check_product_linkage(self):
        order_cols = set(self.staging["orders"].columns)
        has_link = any("product" in c.lower() for c in order_cols)
        self.report_rows.append({
            "rule": "orders <-> products (no linking field)",
            "source": "orders/products",
            "affected_rows": len(self.staging["orders"]),
            "status_action": "Schema gap — no order_items bridge exists; see Task 7",
        })
        return has_link

    def build_report(self):
        return pd.DataFrame(self.report_rows)


checker = IntegrityChecker(staging)
checker.check_fk("orders.customer_id -> customers.customer_id", "orders", "customer_id", "customers", "customer_id")
checker.check_fk("payments.order_id -> orders.order_id", "payments", "order_id", "orders", "order_id")
checker.check_fk("deliveries.order_id -> orders.order_id", "deliveries", "order_id", "orders", "order_id")
checker.check_product_linkage()

data_quality_report = checker.build_report()
data_quality_report

,rule,source,affected_rows,status_action
0,orders.customer_id -> customers.customer_id,orders,0,Reported for review; not silently removed
1,payments.order_id -> orders.order_id,payments,0,Reported for review; not silently removed
2,deliveries.order_id -> orders.order_id,deliveries,0,Reported for review; not silently removed
3,orders <-> products (no linking field),orders/products,15,Schema gap — no order_items bridge exists; see...


## Task 4 - Fulfillment Integrator class

In [36]:
class FulfillmentIntegrator:
    """Builds order_fulfillment_integrated.csv at one-row-per-order grain."""

    def __init__(self, staging, orders_with_derived_fields):
        self.staging = staging
        self.orders = orders_with_derived_fields.copy()
        self.grain_log = []   # tracks row counts before/after each merge

    def _log(self, step, before, after):
        self.grain_log.append({"step": step, "rows_before": before, "rows_after": after,
                                "grain_preserved": before == after})

    def build_delivery_summary(self):
        """Reduce multiple delivery rows per order to one row (grain: order_id)."""
        d = self.staging["deliveries"]
        summary = d.groupby("order_id").agg(
            delivery_count=("delivery_id", "count"),
            latest_delivery_status=("delivery_status", "last"),
            earliest_shipment_date=("shipment_date", "min"),
            latest_delivery_date=("delivery_date", "max"),
        ).reset_index()
        return summary

    def integrate(self):
        before = len(self.orders)

        # 1. Join customers (many orders -> one customer; grain stays at order level)
        result = self.orders.merge(
            self.staging["customers"][["customer_id", "full_name", "email", "city", "customer_type"]],
            on="customer_id", how="left"
        )
        self._log("join customers", before, len(result))

        # 2. Join order-level delivery summary (already aggregated -> 1 row per order)
        delivery_summary = self.build_delivery_summary()
        before = len(result)
        result = result.merge(delivery_summary, on="order_id", how="left")
        self._log("join delivery_summary", before, len(result))

        # is_fully_paid / balance_due / fulfillment_status / successful_amount_paid
        # were already computed at order grain in Task 2 — carried through untouched.
        self.integrated = result
        return self

    def grain_check(self):
        dupes = self.integrated["order_id"].duplicated().sum()
        print(f"order_id duplicates in integrated output: {dupes}")
        print(f"Final row count: {len(self.integrated)} (should equal original orders count)")
        return pd.DataFrame(self.grain_log)


integrator = FulfillmentIntegrator(staging, orders).integrate()
grain_report = integrator.grain_check()
order_fulfillment_integrated = integrator.integrated
grain_report

order_id duplicates in integrated output: 0
Final row count: 15 (should equal original orders count)


,step,rows_before,rows_after,grain_preserved
0,join customers,15,15,True
1,join delivery_summary,15,15,True


## Task 5 - Integration Validator class

In [37]:
class IntegrationValidator:
    """Validates row counts, uniqueness, missingness, and impossible values post-integration."""

    def __init__(self, original_orders, integrated_df, grain_col="order_id"):
        self.original = original_orders
        self.integrated = integrated_df
        self.grain_col = grain_col
        self.findings = []

    def check_row_counts(self):
        before, after = len(self.original), len(self.integrated)
        self.findings.append({"check": "row_count_before_vs_after",
                               "result": f"{before} -> {after}",
                               "status": "OK" if before == after else "REVIEW NEEDED"})

    def check_grain_uniqueness(self):
        dupes = self.integrated[self.grain_col].duplicated().sum()
        self.findings.append({"check": f"{self.grain_col}_uniqueness",
                               "result": f"{dupes} duplicate keys",
                               "status": "OK" if dupes == 0 else "REVIEW NEEDED"})

    def check_new_missing_values(self, join_cols):
        """join_cols: columns introduced by joins, where new NaNs = unmatched parent."""
        for col in join_cols:
            n_missing = self.integrated[col].isna().sum()
            self.findings.append({"check": f"missing_after_join::{col}",
                                   "result": f"{n_missing} nulls",
                                   "status": "OK" if n_missing == 0 else "REVIEW NEEDED"})

    def check_impossible_values(self):
        neg_balance = (self.integrated["balance_due"] < 0).sum()
        neg_total = (self.integrated["total_order_value"] < 0).sum()
        neg_paid = (self.integrated["successful_amount_paid"] < 0).sum()
        self.findings.append({"check": "negative_balance_due", "result": f"{neg_balance} rows", "status": "REVIEW NEEDED" if neg_balance else "OK"})
        self.findings.append({"check": "negative_total_order_value", "result": f"{neg_total} rows", "status": "REVIEW NEEDED" if neg_total else "OK"})
        self.findings.append({"check": "negative_successful_amount_paid", "result": f"{neg_paid} rows", "status": "REVIEW NEEDED" if neg_paid else "OK"})

    def run_all(self, join_cols):
        self.check_row_counts()
        self.check_grain_uniqueness()
        self.check_new_missing_values(join_cols)
        self.check_impossible_values()
        return pd.DataFrame(self.findings)


validator = IntegrationValidator(orders, order_fulfillment_integrated)
validation_report = validator.run_all(join_cols=["full_name", "latest_delivery_status"])
validation_report

,check,result,status
0,row_count_before_vs_after,15 -> 15,OK
1,order_id_uniqueness,0 duplicate keys,OK
2,missing_after_join::full_name,0 nulls,OK
3,missing_after_join::latest_delivery_status,0 nulls,OK
4,negative_balance_due,0 rows,OK
5,negative_total_order_value,0 rows,OK
6,negative_successful_amount_paid,0 rows,OK


In [38]:
# Fold validation findings into the master data-quality report
data_quality_report = pd.concat([data_quality_report, validation_report.rename(
    columns={"check": "rule", "result": "affected_rows", "status": "status_action"})],
    ignore_index=True)
data_quality_report

,rule,source,affected_rows,status_action
0,orders.customer_id -> customers.customer_id,orders,0,Reported for review; not silently removed
1,payments.order_id -> orders.order_id,payments,0,Reported for review; not silently removed
2,deliveries.order_id -> orders.order_id,deliveries,0,Reported for review; not silently removed
3,orders <-> products (no linking field),orders/products,15,Schema gap — no order_items bridge exists; see...
4,row_count_before_vs_after,NaN,15 -> 15,OK
5,order_id_uniqueness,NaN,0 duplicate keys,OK
6,missing_after_join::full_name,NaN,0 nulls,OK
7,missing_after_join::latest_delivery_status,NaN,0 nulls,OK
8,negative_balance_due,NaN,0 rows,OK
9,negative_total_order_value,NaN,0 rows,OK


##Task 6 — Business Queries class

In [39]:
class BusinessQueries:
    """Runs the required Task 6 business validation checks on the integrated output."""

    def __init__(self, integrated_df):
        self.df = integrated_df

    def fully_paid_and_delivered(self):
        result = self.df[(self.df["fulfillment_status"] == "Complete")]
        print(f"Fully paid & delivered orders: {len(result)}")
        return result[["order_id", "full_name", "total_order_value", "balance_due", "fulfillment_status"]]

    def unpaid_or_problem_value(self):
        problem = self.df[self.df["fulfillment_status"] == "Payment Issue"]
        total = problem["balance_due"].sum()
        print(f"Total order value unpaid/pending/failed: {total:,.2f} across {len(problem)} orders")
        return problem[["order_id", "total_order_value", "successful_amount_paid", "balance_due"]]

    def repeat_customers(self, min_orders=2):
        counts = self.df.groupby(["customer_id", "full_name"]).size().reset_index(name="valid_order_count")
        repeats = counts[counts["valid_order_count"] >= min_orders].sort_values("valid_order_count", ascending=False)
        print(f"Repeat customers ({min_orders}+ orders): {len(repeats)}")
        return repeats

    def product_volume_note(self):
        note = ("Product-level order volume cannot be calculated: orders.csv has no "
                "product_id (or any product-referencing column), so there is no field "
                "linking an order to the product(s) it contains. See Task 7 for the "
                "proposed order_items bridge table that would resolve this.")
        print(note)
        return note


bq = BusinessQueries(order_fulfillment_integrated)
q1_complete_orders = bq.fully_paid_and_delivered()
q2_unpaid_value = bq.unpaid_or_problem_value()
q3_repeat_customers = bq.repeat_customers()
q4_product_gap_note = bq.product_volume_note()

Fully paid & delivered orders: 0
Total order value unpaid/pending/failed: 38,411.74 across 15 orders
Repeat customers (2+ orders): 5
Product-level order volume cannot be calculated: orders.csv has no product_id (or any product-referencing column), so there is no field linking an order to the product(s) it contains. See Task 7 for the proposed order_items bridge table that would resolve this.


##Task 7 — Schema Proposer class

In [40]:
class SchemaProposer:
    """Documents the products<->orders schema gap and proposes the order_items bridge table."""

    def __init__(self):
        self.gap_note = None
        self.proposed_schema = None

    def document_gap(self):
        self.gap_note = (
            "orders.csv contains no product_id or any other product-referencing column, "
            "and products.csv contains no order_id or customer-referencing column. "
            "There is no shared key between the two tables, so 'Which products generate "
            "the highest order volume?' cannot be answered from the current five sources. "
            "Any product-level sales figure derived today would be fabricated, not measured."
        )
        print(self.gap_note)
        return self

    def propose_order_items_schema(self):
        self.proposed_schema = pd.DataFrame([
            {"field_name": "order_item_id", "data_type": "object (string)", "key_role": "Primary Key",
             "description": "Unique identifier for a single product line within an order."},
            {"field_name": "order_id", "data_type": "object (string)", "key_role": "Foreign Key -> orders.order_id",
             "description": "Links this line item back to its parent order."},
            {"field_name": "product_id", "data_type": "object (string)", "key_role": "Foreign Key -> products.product_id",
             "description": "Links this line item to the specific product purchased."},
            {"field_name": "quantity", "data_type": "int64", "key_role": "Attribute",
             "description": "Number of units of this product purchased within the order."},
            {"field_name": "unit_price", "data_type": "float64", "key_role": "Attribute",
             "description": "Price per unit at time of purchase (may differ from current products.unit_price)."},
        ])
        return self.proposed_schema


proposer = SchemaProposer().document_gap()
proposed_order_items_schema = proposer.propose_order_items_schema()
proposed_order_items_schema

orders.csv contains no product_id or any other product-referencing column, and products.csv contains no order_id or customer-referencing column. There is no shared key between the two tables, so 'Which products generate the highest order volume?' cannot be answered from the current five sources. Any product-level sales figure derived today would be fabricated, not measured.


,field_name,data_type,key_role,description
0,order_item_id,object (string),Primary Key,Unique identifier for a single product line wi...
1,order_id,object (string),Foreign Key -> orders.order_id,Links this line item back to its parent order.
2,product_id,object (string),Foreign Key -> products.product_id,Links this line item to the specific product p...
3,quantity,int64,Attribute,Number of units of this product purchased with...
4,unit_price,float64,Attribute,Price per unit at time of purchase (may differ...


##Task 8 — Output Writer class

In [41]:
class OutputWriter:
    """Saves Phase 2 deliverables to the output/ folder with index=False."""

    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.saved_files = []

    def save(self, df, filename):
        path = f"{self.output_dir}/{filename}"
        df.to_csv(path, index=False)
        self.saved_files.append({"file": filename, "rows": len(df), "path": path})
        print(f"Saved {path}  ({len(df)} rows)")
        return self

    def summary(self):
        return pd.DataFrame(self.saved_files)


writer = OutputWriter(OUTPUT_DIR)
writer.save(order_fulfillment_integrated, "order_fulfillment_integrated.csv")
writer.save(data_quality_report, "data_quality_report.csv")
writer.save(proposed_order_items_schema, "proposed_order_items_schema.csv")

writer.summary()

Saved /content/drive/MyDrive/data_engineering_project/output/order_fulfillment_integrated.csv  (15 rows)
Saved /content/drive/MyDrive/data_engineering_project/output/data_quality_report.csv  (11 rows)
Saved /content/drive/MyDrive/data_engineering_project/output/proposed_order_items_schema.csv  (5 rows)


,file,rows,path
0,order_fulfillment_integrated.csv,15,/content/drive/MyDrive/data_engineering_projec...
1,data_quality_report.csv,11,/content/drive/MyDrive/data_engineering_projec...
2,proposed_order_items_schema.csv,5,/content/drive/MyDrive/data_engineering_projec...
